# ESM3 alpha dial — prediction

What the dial does to predictive performance on the ESM3 sweep, against the boosting
base on the same folds.

**This notebook draws and does not aggregate** — see the geometry notebook's header for
why that rule is load-bearing. In short: `alpha_grid` averages model seeds inside each
fold and takes the interval over folds, and no figure here recomputes a mean.

Two figures carry the argument. The first shows two absolute levels; the second shows
their **paired difference within each fold**, which is the comparison the paper makes
and the only one whose interval means what it appears to mean.

In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.analysis import alpha_grid as ag

fk.use_house_style()
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/graph/v13_esm3"   # the ESM3 sweep
NODES      = "nodedial"                 # "nodedial" (v9) | "onehot" (v8 gate) | None
MOL_SOURCE = "chemberta"
DATASET    = None          # None -> every dataset on disk; or ["cc", "hc"] / "m2or"
REGIME     = None          # None -> both; or "transductive" / "inductive"
SEED       = None          # model seed; None -> every one present
COMBO      = "cls+mol"     # which head the graph rows come from

CI_LEVEL   = 0.95          # Student-t over FOLDS (seeds are averaged inside each fold)
METRICS    = None          # None -> the metric of record per dataset; or a list
LEVELS     = ("boost_full", "naive")   # horizontal references to draw
RESOLUTION_METRIC = None   # None -> the metric of record of the first series

SAVE_FIGS  = False
FIG_DIR    = "results/article_figures/esm3"
# =============================================================================

df = ag.load(root=ROOT_DIR, mol_source=MOL_SOURCE, nodes=NODES,
             dataset=DATASET, regime=REGIME, seed=SEED, combo=COMBO)

SERIES = sorted(df.series.unique())
ALPHAS = sorted(df.loc[df.arm == "gate", "alpha"].dropna().unique())

In [ ]:
# ---------------------------------------------------------------- what is on disk
# The two dials must never share an axis. On a v8 GATE run alpha mixes the graph's
# OUTPUT against a frozen ESM branch; on a v9 NODE dial it mixes the graph's INPUT,
# and it runs the other way (alpha=0 is receptor identity, alpha=1 the legacy graph).
# A frame holding both puts opposite meanings on one x axis, so refuse it here rather
# than discover it in a plot.
dials = sorted(df["nodes"].astype(str).unique())
if len(dials) != 1:
    raise SystemExit(
        f"this frame holds {dials} -- two dials cannot share an axis. "
        f"Set NODES to one of them.")

cov = ag.coverage(df)
print(f"{len(SERIES)} series | {len(ALPHAS)} alphas | dial: {dials[0]}")
print("alphas:", ", ".join(f"{a:g}" for a in ALPHAS))
pin = ", ".join(f"{k}={v}" for k, v in df.attrs["pinned"].items()) or "-"
print("pinned:", pin)
cov

In [ ]:
MOF = ag.metric_for(df)                     # metric of record per dataset
DS_OF = dict(zip(df.series, df.dataset))
if RESOLUTION_METRIC is None:
    RESOLUTION_METRIC = MOF[DS_OF[SERIES[0]]]
print("metric of record:", MOF)
print("resolution computed on:", RESOLUTION_METRIC)

In [ ]:
# ------------------------------------------------- how small a difference is visible
# The floor: the model noise LEFT IN A FOLD MEAN after averaging the seeds. Every curve
# on this page is made of fold means, so a wiggle below this number is not a finding.
#
# This is also why the curves carry a CI band over folds and an I-mark for the floor,
# and never whiskers over folds: the between-fold spread is the larger quantity by a
# wide margin, and drawing it on the curve answers "how different are the folds" on a
# figure that is asking "did moving the dial change anything".
ns = ag.noise_split(df, RESOLUTION_METRIC)
FLOOR = {}
if len(ns):
    g = ns.groupby("series")[["within", "between", "floor"]].mean()
    FLOOR = g["floor"].to_dict()
    display(g.round(4))
    print("\nratio between/floor -- how much of the spread is the folds, not the model:")
    display((g["between"] / g["floor"]).round(1).rename("x"))
else:
    print(f"{RESOLUTION_METRIC!r} not in this sweep; no floor computed")

---
## 2. Performance along the dial

In [ ]:
# 2.1  THE METRIC OF RECORD ALONG THE DIAL, with the boosting base as a dashed line.
# The band is the CI over folds; the grey I-mark on the left is the resolution floor
# from the cell above -- a bump shorter than that mark is model noise.
fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.3, h=2.5)

for i, s in enumerate(SERIES):
    a = ax[i]
    metric = MOF[DS_OF[s]]
    sub = df[df.series == s]
    cur = ag.curve(sub, metric, by=("series",), level=CI_LEVEL)
    drawn = fk.curve_with_band(a, cur, fk.C["gate"])
    lv = ag.levels(sub, metric, arms=LEVELS, by=("series",), level=CI_LEVEL)
    for _, r in lv.iterrows():
        fk.reference_line(a, r["mean"], r["arm"])
    fk.alpha_axis(a, ALPHAS)
    if not drawn:
        fk.note_empty(a)
    else:
        fk.resolution_mark(a, FLOOR.get(s))
    a.set_title(s, fontsize=8.5)
    a.set_xlabel(r"$\alpha$")
    a.set_ylabel(metric)

fig.suptitle("The metric of record along the dial", y=1.005, fontsize=11)
fk.figlegend(fig, [
    Line2D([], [], color=fk.C["gate"], marker="o", label="our graph at this $\\alpha$"),
    Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
           label="boosting base (prot+mol)"),
    Line2D([], [], color=fk.C["naive"], ls=fk.DASH["naive"], label="naive (train mean)"),
    Line2D([], [], color=fk.MUTED, lw=6, alpha=0.2,
           label=f"band = {CI_LEVEL:.0%} CI over folds"),
], ncol=4)
fk.savefig(fig, "prediction_curves", FIG_DIR, SAVE_FIGS)
plt.show()

### 2.2 The advantage over the base, paired

In [ ]:
# 2.2  THE ADVANTAGE OVER THE BASE, PAIRED WITHIN EACH FOLD. The curve above shows two
# absolute levels; this shows their difference computed fold by fold, which is the
# comparison the paper actually makes. Zero is the base. A band that clears zero along
# a stretch of the dial is the claim; a band straddling it is not.
fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.3, h=2.5)

for i, s in enumerate(SERIES):
    a = ax[i]
    metric = MOF[DS_OF[s]]
    d = ag.delta_vs(df[df.series == s], metric, ref_arm="boost_full",
                    by=("series",), level=CI_LEVEL)
    a.axhline(0.0, color=fk.C["boost_full"], lw=1.3, ls=fk.DASH["boost_full"])
    drawn = fk.curve_with_band(a, d, fk.C["fun"])
    fk.alpha_axis(a, ALPHAS)
    if not drawn:
        fk.note_empty(a)
    a.set_title(s, fontsize=8.5)
    a.set_xlabel(r"$\alpha$")
    a.set_ylabel(f"{metric} - base")

fig.suptitle("Paired advantage over the boosting base, fold by fold", y=1.005,
             fontsize=11)
fk.figlegend(fig, [
    Line2D([], [], color=fk.C["fun"], marker="o", label="our graph minus the base"),
    Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"], label="the base"),
    Line2D([], [], color=fk.MUTED, lw=6, alpha=0.2,
           label=f"band = {CI_LEVEL:.0%} CI over folds"),
], ncol=3)
fk.savefig(fig, "prediction_delta", FIG_DIR, SAVE_FIGS)
plt.show()

### 2.3 The whole battery

In [ ]:
# 2.3  THE WHOLE BATTERY, not just the metric of record. A gain that shows up in one
# metric and nowhere else is a property of that metric.
metrics = METRICS or ag.metrics_available(df, which="headline")
metrics = [m for m in metrics if m in df.columns]
print("drawing:", metrics)

for s in SERIES:
    sub = df[df.series == s]
    have = [m for m in metrics if pd.to_numeric(sub[m], errors="coerce").notna().any()]
    if not have:
        continue
    fig, ax = fk.panels(len(have), ncols=min(4, len(have)), w=2.7, h=2.1, sharex=True)
    for j, m in enumerate(have):
        a = ax[j]
        fk.curve_with_band(a, ag.curve(sub, m, by=("series",), level=CI_LEVEL),
                           fk.C["gate"])
        lv = ag.levels(sub, m, arms=LEVELS, by=("series",), level=CI_LEVEL)
        for _, r in lv.iterrows():
            fk.reference_line(a, r["mean"], r["arm"])
        fk.alpha_axis(a, ALPHAS)
        a.set_title(m, fontsize=8.5)
        a.set_xlabel(r"$\alpha$")
    fig.suptitle(s, y=1.01, fontsize=10.5)
    fk.figlegend(fig, [
        Line2D([], [], color=fk.C["gate"], marker="o", label="our graph"),
        Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"], label="base"),
        Line2D([], [], color=fk.C["naive"], ls=fk.DASH["naive"], label="naive"),
    ], ncol=3)
    fk.savefig(fig, f"prediction_battery_{s.replace(' / ', '_').replace(' ', '')}",
               FIG_DIR, SAVE_FIGS)
    plt.show()

---
## Before quoting any alpha from this page

Choosing an alpha on these curves and then reporting the same curves is choosing the
number and its defence from the same rows. The honest form is
`scripts/analysis/val_rescore.py` followed by `alpha_choice.py --select-on val`, which
picks on validation and reads test once. `notebooks/graph/node_dial/` does that and is
where a reported alpha should come from.